# **Qwen2.5-omni**

## 1.환경준비

### (1) 라이브러리 설치

In [ ]:
!pip -q install -U "transformers>=4.52.0" accelerate qwen-omni-utils[decord] soundfile

### (2) 라이브러리 로딩

In [ ]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

## 2.Qwen 사용해보기

### (1) 모델 다운로드
* 모델 다운로드 : 5~8분

In [ ]:
model_id = "Qwen/Qwen2.5-Omni-3B"  # 경량
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

### (2) 모델 사용하기

* 파일 준비

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 beach_children.jpg 선택

* 모델 사용

In [ ]:
USE_AUDIO_IN_VIDEO = False  # 영상의 내부 오디오까지 쓸지 여부
USE_AUDIO = False

conversation = [
    {"role":"system","content":[{"type":"text","text":"You are a helpful assistant."}]},
    {"role":"user","content":[
        {"type":"image","image":"beach_children.jpg"},
        {"type":"text","text":"사람 수와 상황을 설명해줘."}
    ]}
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)

inputs = processor(
    text=text, audio=audios, images=images, videos=videos,
    return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO
).to(model.device)

gen_kwargs = dict(
    **inputs,
    return_audio=USE_AUDIO,
    use_audio_in_video=False,
    max_new_tokens=256,
)

with torch.inference_mode():
    out = model.generate(**gen_kwargs)

# 출력 분기 처리
if USE_AUDIO:
    text_ids, audio = out            # 음성까지 요청한 경우: (text_ids, audio) 튜플
else:
    text_ids = out                   # 텍스트만 요청한 경우: 텐서 1개

print(processor.batch_decode(text_ids, skip_special_tokens=True)[0])

### (3) 실습
* 다양한 이미지를 다운받아, 모델에 입력하고, 사용해 봅시다.